### Check GPU Status
This cell checks the status of the NVIDIA GPU available in the environment.

**Why this step:** Training a YOLOv8 object-detection model on 1,000+ images is computationally infeasible on CPU. Before writing any training code, we verify that Colab has actually allocated a GPU (`Runtime → Change runtime type → GPU`), so we don't discover mid-run that training is silently crawling on CPU.

In [ ]:
!nvidia-smi

### Install and Import Libraries
This cell installs the `ultralytics` and `roboflow` libraries, imports the necessary modules, and checks CUDA availability.

**Why this step:** The GPU check above confirmed hardware is available; now we install the two libraries the whole pipeline depends on — `ultralytics` (the YOLOv8 implementation) and `roboflow` (dataset hosting/download). The explicit `torch.cuda.is_available()` check verifies that PyTorch can actually *use* the GPU we saw in `nvidia-smi` — a mismatch here would mean a broken CUDA installation.

In [ ]:
!pip install ultralytics roboflow -q

from ultralytics import YOLO
from roboflow import Roboflow
import torch

print("CUDA available:", torch.cuda.is_available())

### Download Dataset from Roboflow

This section downloads the dataset directly from Roboflow using the provided API key, workspace, project, and version.

**Why this step:** Instead of manually uploading a ZIP file, we are leveraging the Roboflow API for a direct and automated dataset download. Note that the code deliberately queries the project's available versions and downloads the **latest** one, so the version can change between re-runs (this run downloaded **v5**, saved to `/content/wheelchair-5`) — always read the actual version and path from the printed output below, and be aware that results may shift if the dataset gains a new version between runs.

In [ ]:
import roboflow
import os

# Dataset: Roboflow wheelchair-9qvfx-bchvo
ROBOFLOW_API_KEY = "wBuO0Xm5iCujzAp7ZaZj"
ROBOFLOW_WORKSPACE = "first-group-project"
ROBOFLOW_PROJECT = "wheelchair-9qvfx-bchvo"
ROBOFLOW_MODEL_FORMAT = "yolov8"

rf = roboflow.Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)

# Get all versions and find the latest one
all_versions = project.versions()
latest_version_number = max([v.version for v in all_versions])

print(f"Downloading dataset from Roboflow: {ROBOFLOW_PROJECT} v{latest_version_number}")

version = project.version(latest_version_number)
dataset = version.download(ROBOFLOW_MODEL_FORMAT)

print(f"\nDataset downloaded to: {dataset.location}")
print("Listing contents of the downloaded directory:")
!ls -R {dataset.location}

### Important: Verify Dataset Structure and `data.yaml`

After download, confirm that `data.yaml` and the `train` / `valid` / `test` folders live directly under `dataset.location` (the exact folder name depends on the version actually downloaded — e.g. `/content/wheelchair-5` for v5 — so read it from `dataset.location` rather than assuming a fixed path).

The next cell loads `data.yaml` and checks that each split path resolves to a real folder of images.

In [ ]:
import yaml
import os
from pathlib import Path

data_root = Path(dataset.location)
data_yaml_path = data_root / "data.yaml"

if not data_yaml_path.exists():
    raise FileNotFoundError(
        f"'data.yaml' not found in {data_root}. Check the extracted folder structure."
    )

with open(data_yaml_path) as f:
    data_cfg = yaml.safe_load(f)

# Correct the paths in data_cfg to be relative to the dataset.location
# The downloaded data.yaml might have paths like ../train/images which resolve incorrectly
# We want them to be relative to the data_root itself (e.g., train/images)
for split in ["train", "val", "test"]:
    if data_cfg.get(split) and data_cfg[split].startswith("../"):
        data_cfg[split] = data_cfg[split].replace("../", "") # Remove the incorrect '..' prefix

# Save the corrected data_cfg back to data.yaml
with open(data_yaml_path, "w") as f:
    yaml.dump(data_cfg, f)

print("Successfully loaded and corrected data.yaml:")
print(yaml.dump(data_cfg, sort_keys=False, default_flow_style=False))

print("Split path checks:")
for split in ["train", "val", "valid", "test"]:
    rel = data_cfg.get(split)
    if rel is None:
        continue
    # The resolved path should now correctly point within data_root
    resolved = (data_root / rel).resolve()
    exists = resolved.exists()
    n_imgs = len(list(resolved.glob("*"))) if exists and resolved.is_dir() else 0
    print(f"  {split}: {rel} -> {resolved} | exists={exists} | files={n_imgs}")

print("\nOn-disk folders under dataset.location:")
for split in ["train", "valid", "test"]:
    img_dir = data_root / split / "images"
    lbl_dir = data_root / split / "labels"
    n_img = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*"))) if lbl_dir.exists() else 0
    print(f"  {split}: images={n_img}, labels={n_lbl}")

### Dataset Analysis

In [ ]:
import os
from collections import Counter

# Get image counts for each split
split_counts = {}
for split in ['train', 'valid', 'test']:
    split_image_path = os.path.join(dataset.location, split, 'images')
    if os.path.exists(split_image_path):
        split_counts[split] = len(os.listdir(split_image_path))
    else:
        split_counts[split] = 0
        print(f"Warning: {split}/images directory not found at {split_image_path}")

total_images = sum(split_counts.values())

print(f"Total images in the dataset: {total_images}\n")

print("Image distribution by split:")
for split, count in split_counts.items():
    percentage = (count / total_images) * 100 if total_images > 0 else 0
    print(f"  {split.capitalize()}: {count} images ({percentage:.2f}%) ")

# Count objects per class across all splits
class_counts = Counter()
for split in ['train', 'valid', 'test']:
    split_label_path = os.path.join(dataset.location, split, 'labels')
    if os.path.exists(split_label_path):
        for label_file in os.listdir(split_label_path):
            with open(os.path.join(split_label_path, label_file), 'r') as f:
                for line in f:
                    if line.strip(): # Ensure line is not empty
                        cls_id = int(line.split()[0])
                        class_counts[cls_id] += 1
    else:
        print(f"Warning: {split}/labels directory not found at {split_label_path}")

print("\nObjects per class (across all splits):")
class_names = data_cfg.get('names', [])
if class_names:
    for cls_id in sorted(class_counts.keys()):
        name = class_names[cls_id] if cls_id < len(class_names) else f"Unknown Class ({cls_id})"
        print(f"  {name}: {class_counts[cls_id]} objects")
else:
    print("  Class names not found in data_cfg.")


### Load Dataset Configuration
This cell loads `data.yaml` from the downloaded dataset to retrieve class names and counts, and prints image counts for each split (train, valid, test).

**Why this step:** Before training anything, we need to know exactly what we downloaded: how many classes, what their names are, and how many images exist per split. Read the counts from the cell's own printed output below rather than assuming a fixed number — the split sizes (and even whether a `test` folder exists) can differ between dataset versions.

In [ ]:
import yaml

with open(f"{dataset.location}/data.yaml") as f:
    data_cfg = yaml.safe_load(f)

print("Number of classes:", data_cfg["nc"])
print("Class names:", data_cfg["names"])
print()

import os
for split in ["train", "valid", "test"]:
    img_dir = f"{dataset.location}/{split}/images"
    if os.path.exists(img_dir):
        count = len(os.listdir(img_dir))
        print(f"{split}: {count} images")
    else:
        print(f"{split}: folder not found")

### Check Validation Data Availability
Confirm that the downloaded dataset ships with a built-in `valid` split (`valid/images` + `valid/labels`).

**Why this step:** All evaluations, model comparisons, and error analysis in this notebook (`model.val(split="valid")` and friends) use this split. The `test` split is intentionally left untouched through all of this — data prep, training, hyperparameter selection, and the run comparison — so it stays available as a fully held-out set. It is used exactly once, in the "Final Held-Out Check on the Test Split" cell near the end of the notebook, after a model has already been chosen.

### Using the Dataset's Own Validation Set

This notebook trains and evaluates on **Roboflow `wheelchair-9qvfx-bchvo`** (see the "Download Dataset" cell for the exact version downloaded this run), which already includes `train` / `valid` / `test`.

Skip any older workflow that built a custom `test_clean` / `test_clean_filtered` set (that was for a previous dataset without a test split). From here on, every evaluation — training-time validation, the run-comparison table, and the error/failure analysis — uses the dataset's native `valid` folder via `data.yaml`. The `test` split is deliberately not used anywhere until the single, dedicated held-out check near the end of the notebook, so every decision made before that point remains uninformed by it.

### Visual Sanity Check on the Validation Set
This cell draws ground-truth boxes on a random sample of images from the dataset's `valid` split.

**Why this step:** A quick visual check — do the drawn boxes and class labels match the image? — is cheap insurance before trusting metrics computed on this split.

In [ ]:
# ============================================================
# Visual sanity check: draw bounding boxes on a sample of test
# images to confirm the class mapping is correct
# ============================================================

import random
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

data_root = Path(dataset.location)
valid_images_dir = data_root / "valid" / "images"
valid_labels_dir = data_root / "valid" / "labels"

class_names = data_cfg["names"]  # e.g. ['people_wheelchair', 'person', 'wheelchair']
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]  # BGR: red, green, blue

all_images = sorted(valid_images_dir.iterdir())
random.seed(0)
sample = random.sample(all_images, min(9, len(all_images)))

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

for ax, img_path in zip(axes, sample):
    label_path = valid_labels_dir / (img_path.stem + ".txt")
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    if label_path.exists():
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])

                x1 = int((xc - bw / 2) * w)
                y1 = int((yc - bh / 2) * h)
                x2 = int((xc + bw / 2) * w)
                y2 = int((yc + bh / 2) * h)

                color = colors[cls_id % len(colors)]
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                label_text = class_names[cls_id]
                cv2.putText(img, label_text, (x1, max(y1 - 10, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Class colors legend:")
for i, name in enumerate(class_names):
    print(f"  {name}: color index {i} ({colors[i % len(colors)]} in BGR)")


### Analyze Class Distribution in Training Data
This cell calculates and prints the distribution of object classes in the training set's label files.

**Why this step:** With the evaluation data sorted out, we turn to the training data. Object detectors struggle with rare classes, and the right augmentation strategy depends on *how* imbalanced the data is. Counting instances per class tells us whether we need imbalance-compensation techniques (like `copy_paste`) before we commit GPU-hours to a long training run.

### Check for Empty Label Files
This cell iterates through the training labels directory to identify and count any empty label files.

**Why this step:** The class-distribution count above only looks at labels that exist. Empty label files are a different failure mode — images that were uploaded but never annotated. If present in large numbers, they would teach the model that objects it should detect are "background". We check this now, before training, because it is far cheaper to fix data than to debug a mysteriously under-performing model later.

In [ ]:
import os

train_labels = f"{dataset.location}/train/labels"
train_images = f"{dataset.location}/train/images"

empty_labels = []
for label_file in os.listdir(train_labels):
    path = os.path.join(train_labels, label_file)
    if os.path.getsize(path) == 0:
        empty_labels.append(label_file)

print(f"Images with empty label files: {len(empty_labels)} out of {len(os.listdir(train_images))}")
print(f"Percentage: {len(empty_labels) / len(os.listdir(train_images)) * 100:.1f}%")

### Visualize Images with Empty Label Files
This cell displays a sample of the images that were found to have corresponding empty label files. These images might indicate missing annotations that could affect model training.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random
import os

# Ensure train_images and empty_labels are available from the previous cell
if 'train_images' not in locals() or 'empty_labels' not in locals():
    print("Error: 'train_images' or 'empty_labels' not found. Please run the previous cell first.")
else:
    print(f"Displaying up to 9 images with empty label files (total: {len(empty_labels)}):")

    # Take a random sample if there are more than 9 empty label files
    sample_empty_images = random.sample(empty_labels, min(len(empty_labels), 9))

    if sample_empty_images:
        fig, axes = plt.subplots(3, 3, figsize=(15, 15))
        axes = axes.flatten()

        for i, label_filename in enumerate(sample_empty_images):
            # Construct the image filename from the label filename
            img_filename = os.path.splitext(label_filename)[0] + '.jpg' # Assuming images are .jpg
            img_path = os.path.join(train_images, img_filename)

            # Check for other common image extensions if .jpg not found
            if not os.path.exists(img_path):
                img_filename = os.path.splitext(label_filename)[0] + '.png'
                img_path = os.path.join(train_images, img_filename)

            if os.path.exists(img_path):
                img = mpimg.imread(img_path)
                axes[i].imshow(img)
                axes[i].set_title(f'Empty Label: {label_filename}', fontsize=10)
                axes[i].axis('off')
            else:
                axes[i].set_title(f'Image not found for {label_filename}', fontsize=10)
                axes[i].axis('off')

        # Turn off any unused subplots
        for j in range(len(sample_empty_images), len(axes)):
            axes[j].axis('off')

        plt.tight_layout()
        plt.show()
    else:
        print("No empty label files to display.")

### Visualize Sample Images with Ground-Truth Boxes
This cell defines a helper that draws bounding boxes on images, then displays a sample of *training* images with their annotations.

**Why this step:** The two previous checks were numeric (counts, empty files). This one is qualitative: are the training annotations actually *correct and tight*? Sloppy boxes or wrong classes in training data would put a hard ceiling on model quality no matter how well we tune hyperparameters. This completes our pre-training data audit.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob
import random
import os

def draw_yolo_boxes(img_path, label_path, class_names, ax):
    img = mpimg.imread(img_path)
    h, w = img.shape[0], img.shape[1]
    ax.imshow(img)
    colors = ['#00FF00', '#FF3333', '#3399FF']
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                cls, xc, yc, bw, bh = map(float, line.split())
                cls = int(cls)
                x0 = (xc - bw / 2) * w
                y0 = (yc - bh / 2) * h
                rect_w = bw * w
                rect_h = bh * h
                rect = plt.Rectangle((x0, y0), rect_w, rect_h, fill=False,
                                      edgecolor=colors[cls % len(colors)], linewidth=2)
                ax.add_patch(rect)
                ax.text(x0, max(y0 - 5, 0), class_names[cls], color=colors[cls % len(colors)],
                        fontsize=9, weight='bold')
    ax.axis('off')

train_imgs = sorted(glob.glob(f"{dataset.location}/train/images/*"))
sample_imgs = random.sample(train_imgs, min(6, len(train_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, sample_imgs):
    label_path = img_path.replace('images', 'labels')
    label_path = os.path.splitext(label_path)[0] + '.txt'
    draw_yolo_boxes(img_path, label_path, data_cfg["names"], ax)

plt.suptitle('Sample training images with ground-truth boxes')
plt.tight_layout()
plt.show()

### Detect Outlier Bounding Box Dimensions using IQR
This cell performs outlier detection on the bounding box dimensions (width and height) across all splits of the dataset (train, valid, test). It uses the Interquartile Range (IQR) method to identify boxes with dimensions significantly different from the majority. This helps in understanding the distribution of object sizes and detecting potential annotation errors or unusual object scales that might affect model training.

**Why this step:** Models can sometimes struggle with extremely small or large bounding boxes, or with inconsistent annotations. Identifying outliers in bounding box dimensions can highlight data quality issues or unusual object characteristics that may require special handling (e.g., specific augmentation strategies or closer inspection of annotations).

In [ ]:
import numpy as np

def calculate_iqr_outliers(data, threshold=1.5):
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - threshold * IQR
    upper_bound = Q3 + threshold * IQR
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    return outliers, lower_bound, upper_bound

# Collect all bounding box widths and heights
all_widths = []
all_heights = []

for split in ['train', 'valid', 'test']:
    split_label_path = os.path.join(dataset.location, split, 'labels')
    if os.path.exists(split_label_path):
        for label_file in os.listdir(split_label_path):
            with open(os.path.join(split_label_path, label_file), 'r') as f:
                for line in f:
                    if line.strip():
                        parts = line.split()
                        if len(parts) >= 5:
                            # YOLO format: class x_center y_center width height
                            width = float(parts[3])
                            height = float(parts[4])
                            all_widths.append(width)
                            all_heights.append(height)

all_widths = np.array(all_widths)
all_heights = np.array(all_heights)

print("--- Bounding Box Dimension Outlier Analysis ---")

# Analyze widths
width_outliers, w_lower, w_upper = calculate_iqr_outliers(all_widths)
print(f"\nWidth Analysis (total boxes: {len(all_widths)}):")
print(f"  Min: {np.min(all_widths):.4f}, Max: {np.max(all_widths):.4f}")
print(f"  Median: {np.median(all_widths):.4f}")
print(f"  Q1: {np.percentile(all_widths, 25):.4f}, Q3: {np.percentile(all_widths, 75):.4f}")
print(f"  IQR Bounds: [{w_lower:.4f}, {w_upper:.4f}]")
print(f"  Number of width outliers: {len(width_outliers)} ({len(width_outliers)/len(all_widths)*100:.2f}%) ")

# Analyze heights
height_outliers, h_lower, h_upper = calculate_iqr_outliers(all_heights)
print(f"\nHeight Analysis (total boxes: {len(all_heights)}):")
print(f"  Min: {np.min(all_heights):.4f}, Max: {np.max(all_heights):.4f}")
print(f"  Median: {np.median(all_heights):.4f}")
print(f"  Q1: {np.percentile(all_heights, 25):.4f}, Q3: {np.percentile(all_heights, 75):.4f}")
print(f"  IQR Bounds: [{h_lower:.4f}, {h_upper:.4f}]")
print(f"  Number of height outliers: {len(height_outliers)} ({len(height_outliers)/len(all_heights)*100:.2f}%) ")

# Combined outliers (any box that is an outlier in either width or height)
combined_outliers_count = 0
for w, h in zip(all_widths, all_heights):
    if (w < w_lower or w > w_upper) or (h < h_lower or h > h_upper):
        combined_outliers_count += 1

print(f"\nTotal unique bounding boxes with at least one outlier dimension: {combined_outliers_count} ({combined_outliers_count/len(all_widths)*100:.2f}%)")

### Visualize the Bounding-Box Outliers Found Earlier
The IQR outlier check earlier in the notebook only printed summary statistics (counts and percentages) for bounding-box width/height outliers, without showing which images they come from. This cell re-runs the same IQR logic but keeps track of the source image/label for every outlier box, then displays a sample of them with the outlier box drawn in red against the image.

**Why this step:** A width/height number alone doesn't tell you whether an outlier is a genuine annotation error (e.g. a box covering the whole image, or a sliver a few pixels wide) or a legitimate very-small/very-large object. Seeing the actual images makes that call possible.

In [ ]:
import os
import cv2
import math
import numpy as np
import matplotlib.pyplot as plt

def calculate_iqr_bounds(data, threshold=1.5):
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)
    IQR = Q3 - Q1
    return Q1 - threshold * IQR, Q3 + threshold * IQR

# Re-collect widths/heights, but this time keep (split, image_file, class_id, box) alongside each value
records = []
for split in ["train", "valid", "test"]:
    split_label_path = os.path.join(dataset.location, split, "labels")
    split_image_path = os.path.join(dataset.location, split, "images")
    if not os.path.exists(split_label_path):
        continue
    for label_file in os.listdir(split_label_path):
        with open(os.path.join(split_label_path, label_file)) as f:
            for line in f:
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                width, height = float(parts[3]), float(parts[4])
                img_stem = os.path.splitext(label_file)[0]
                records.append({
                    "split": split, "img_stem": img_stem, "img_dir": split_image_path,
                    "cls_id": cls_id, "width": width, "height": height,
                })

widths = np.array([r["width"] for r in records])
heights = np.array([r["height"] for r in records])
w_lower, w_upper = calculate_iqr_bounds(widths)
h_lower, h_upper = calculate_iqr_bounds(heights)

outlier_records = [
    r for r, w, h in zip(records, widths, heights)
    if (w < w_lower or w > w_upper) or (h < h_lower or h > h_upper)
]
print(f"Total outlier boxes: {len(outlier_records)} / {len(records)} "
      f"({len(outlier_records)/len(records)*100:.2f}%)")

# Rank by how far outside the bound they fall, show the most extreme ones
def severity(r):
    d = 0.0
    if r["width"] < w_lower: d += (w_lower - r["width"])
    if r["width"] > w_upper: d += (r["width"] - w_upper)
    if r["height"] < h_lower: d += (h_lower - r["height"])
    if r["height"] > h_upper: d += (r["height"] - h_upper)
    return d

outlier_records.sort(key=severity, reverse=True)
sample = outlier_records[:12]

names_local = data_cfg["names"]
cols = 3
rows = math.ceil(len(sample) / cols) if sample else 1
fig, axes = plt.subplots(rows, cols, figsize=(16, 5.5 * rows))
axes = np.array(axes).reshape(-1)

for idx, r in enumerate(sample):
    img_candidates = [p for p in os.listdir(r["img_dir"]) if p.startswith(r["img_stem"])]
    if not img_candidates:
        axes[idx].axis("off")
        continue
    img_path = os.path.join(r["img_dir"], img_candidates[0])
    img = cv2.imread(img_path)
    if img is None:
        axes[idx].axis("off")
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h_img, w_img = img.shape[:2]

    label_path = os.path.join(r["img_dir"].replace("images", "labels"), r["img_stem"] + ".txt")
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                is_this_outlier = (
                    cls_id == r["cls_id"] and abs(bw - r["width"]) < 1e-6 and abs(bh - r["height"]) < 1e-6
                )
                x1 = int((xc - bw / 2) * w_img); y1 = int((yc - bh / 2) * h_img)
                x2 = int((xc + bw / 2) * w_img); y2 = int((yc + bh / 2) * h_img)
                color = (255, 0, 0) if is_this_outlier else (100, 100, 100)
                thickness = 3 if is_this_outlier else 1
                cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)

    axes[idx].imshow(img)
    axes[idx].set_title(
        f"{r['split']}/{r['img_stem']}\n{names_local[r['cls_id']]}  w={r['width']:.3f} h={r['height']:.3f}",
        fontsize=9,
    )
    axes[idx].axis("off")

for idx in range(len(sample), len(axes)):
    axes[idx].axis("off")

plt.suptitle("Most extreme bounding-box outliers (red = the outlier box itself)", fontsize=13)
plt.tight_layout()
plt.show()


### Initialize YOLOv8 Model
This cell loads a pre-trained YOLOv8 **small** model (`yolov8s.pt`) to be fine-tuned on our dataset.

**Why this step:** The data checks all passed, so we can start modeling. We deliberately begin with the *small* variant, not the strongest one: the goal of the first run is to validate that the whole pipeline works end-to-end (data loads, losses decrease, metrics are produced), and a small model does that in a fraction of the time. Transfer learning from COCO-pretrained weights means we start from strong general visual features instead of random initialization.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8s.pt")

### Train the YOLOv8 Model (Baseline)
This cell runs a short baseline training: `epochs=30`, `imgsz=640`, `batch=-1` (auto batch size), `patience=20`.

**Why this step:** Before investing in the heavier full-model runs, we need proof that nothing in the pipeline is broken. A 30-epoch run with the small model is our smoke test: if class names are wrong, labels are malformed, or CUDA misbehaves, we find out here at minimal cost. Only after this baseline runs cleanly do we move to the full-size experiments.

In [ ]:
model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,
    imgsz=640,
    batch=-1,
    patience=20
)

### Quantify Class Imbalance Before Full Training
This cell counts the exact number of instances per class in the training labels and prints each class's share of the total.

**Why this step:** The baseline run completed without errors, confirming the pipeline works. Before configuring the *full* training run, we return to the imbalance question with exact numbers: the count reveals that `person` dominates while `wheelchair` and `people_wheelchair` are much rarer. This number directly determines our next decision — enabling `copy_paste` augmentation, whose recommended range (0.3–0.5) is meant precisely for compensating rare classes.

In [ ]:
from collections import Counter
import os

label_dir = f"{dataset.location}/train/labels"
counts = Counter()
for f in os.listdir(label_dir):
    with open(os.path.join(label_dir, f)) as file:
        for line in file:
            cls_id = int(line.split()[0])
            counts[cls_id] += 1

names = data_cfg["names"]
total = sum(counts.values())
for cls_id, cnt in sorted(counts.items()):
    print(f"{names[cls_id]}: {cnt} ({cnt/total*100:.1f}%)")

### Train YOLOv8m — Attempt 1 (`copy_paste=0.4`)
This cell runs the first full training of the **medium** model: `epochs=30`, `patience=30`, `cache='ram'`, `copy_paste=0.4`.

**Why this step:** Two decisions flow from what we just learned. First, we upgrade from `yolov8s` to `yolov8m`: the pipeline is proven, so we can afford the stronger model, which offers the best speed/accuracy balance for our deployment goal. Second, the class count showed real imbalance, so we enable `copy_paste=0.4` — the middle of the recommended 0.3–0.5 range — which duplicates objects from rare classes into other training images. `patience=30` gives early stopping so we don't waste GPU time once improvement plateaus; `seed=0` makes runs comparable.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8m.pt")

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,
    patience=30,
    cache="ram",
    device=0,
    amp=True,
    seed=0,
    copy_paste=0.4   # in the middle of the range 0.3-0.5, compensates for the two rare classes
)

### Visualize Training Results (Attempt 1)
This cell displays the loss curves, metric curves, and normalized confusion matrix for the first YOLOv8m run.

**Why this step:** We never change a hyperparameter blindly — every next decision must be grounded in what this run actually shows. The loss curves tell us whether we are over- or under-fitting; the confusion matrix tells us *which classes* the model mixes up. The findings here (rare-class performance and a notable `people_wheelchair` ↔ `wheelchair` confusion) will drive the next two experiments.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Update the path if your run was saved elsewhere
run_dir = "/content/runs/detect/train-2"

# Show results.png (loss, mAP, precision/recall curves over epochs)
img_results = mpimg.imread(f"{run_dir}/results.png")
plt.figure(figsize=(16, 10))
plt.imshow(img_results)
plt.axis("off")
plt.title("Training Results")
plt.show()

# Show confusion_matrix.png (normalized version - easier to read as percentages)
img_cm = mpimg.imread(f"{run_dir}/confusion_matrix_normalized.png")
plt.figure(figsize=(10, 8))
plt.imshow(img_cm)
plt.axis("off")
plt.title("Confusion Matrix (Normalized)")
plt.show()

### Train YOLOv8m — Attempt 2 (`copy_paste=0.5`)
This cell repeats the training with a single change: `copy_paste` raised from `0.4` to `0.5`.

**Why this step:** Attempt 1 showed the rare classes could still improve, and 0.4 sits in the middle of the recommended range — so the natural question is whether the top of the range (0.5) helps further. Crucially, we change **only one parameter at a time**, so any difference in results can be attributed to `copy_paste` alone. This one-variable-at-a-time discipline is applied throughout the rest of the notebook.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8m.pt")

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,
    patience=30,
    cache="ram",
    device=0,
    amp=True,
    seed=0,
    copy_paste=0.5   # in the middle of the range 0.3-0.5, compensates for the two rare classes
)

### Visualize Training Results (Attempt 2)
This cell plots the performance graphs and confusion matrix for the second run.

**Why this step:** We need the verdict on the 0.4 → 0.5 change before deciding what to try next. The comparison shows the results are essentially unchanged between the two values — meaning `copy_paste` is saturated as a lever. However, the confusion matrix still shows a substantial problem: a large share of `people_wheelchair` instances are misclassified as `wheelchair` (read the exact rate from the normalized confusion matrix printed below, rather than assuming a fixed percentage). That specific finding redirects our attention from augmentation to the classification loss itself.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Update the path if your run was saved elsewhere
run_dir = "/content/runs/detect/train-3"

# Show results.png (loss, mAP, precision/recall curves over epochs)
img_results = mpimg.imread(f"{run_dir}/results.png")
plt.figure(figsize=(16, 10))
plt.imshow(img_results)
plt.axis("off")
plt.title("Training Results")
plt.show()

# Show confusion_matrix.png (normalized version - easier to read as percentages)
img_cm = mpimg.imread(f"{run_dir}/confusion_matrix_normalized.png")
plt.figure(figsize=(10, 8))
plt.imshow(img_cm)
plt.axis("off")
plt.title("Confusion Matrix (Normalized)")
plt.show()

### Train YOLOv8m — Attempt 3 (`cls=0.8`)
This cell keeps `copy_paste=0.5` and raises the classification loss weight from its default `0.5` to `cls=0.8`.

**Why this step:** The previous diagnosis showed the bottleneck is no longer augmentation but *class confusion* — the model localizes objects well but sometimes labels `people_wheelchair` as `wheelchair` instead (see the normalized confusion matrix in the previous cell for the exact rate). The `cls` parameter controls how heavily classification errors are penalized relative to localization errors, so increasing it is the targeted response to exactly this failure mode. `copy_paste=0.5` is kept unchanged because it did no harm and the rare classes still benefit from it.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8m.pt")

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,
    patience=30,
    cache="ram",
    device=0,
    amp=True,
    seed=0,
    copy_paste=0.5,   # remains - helps with identification, even if the result did not change between 0.4-0.5
    cls=0.8            # from the original 0.5, increases the classification loss weight -> less class confusion
)

### Visualize Training Results (Attempt 3)
This cell displays the training curves and confusion matrix for the third attempt.

**Why this step:** We check whether the `cls=0.8` intervention hit its target. Compare the `people_wheelchair` → `person` confusion rate in the normalized confusion matrix below against the equivalent cell for Attempt 2 — do not assume a specific percentage until you have read both matrices, since earlier drafts of this notebook stated numbers here that did not match the actual printed confusion matrices. The obvious follow-up question: if `cls=0.8` helped, would pushing further help more?

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Update the path if your run was saved elsewhere (check the "save_dir" printed above)
run_dir = "/content/runs/detect/train-4"

# Show results.png (loss, mAP, precision/recall curves over epochs)
img_results = mpimg.imread(f"{run_dir}/results.png")
plt.figure(figsize=(16, 10))
plt.imshow(img_results)
plt.axis("off")
plt.title("Training Results")
plt.show()

# Show confusion_matrix.png (normalized version - easier to read as percentages)
img_cm = mpimg.imread(f"{run_dir}/confusion_matrix_normalized.png")
plt.figure(figsize=(10, 8))
plt.imshow(img_cm)
plt.axis("off")
plt.title("Confusion Matrix (Normalized)")
plt.show()

### Train YOLOv8m — Attempt 4 (`cls=1.0`)
This cell runs a fourth iteration with the classification weight pushed further, to `cls=1.0`.

**Why this step:** Attempt 3 showed that raising `cls` from 0.5 to 0.8 reduced the target confusion (compare the two normalized confusion matrices directly — do not assume a specific percentage without reading both). Following the same one-parameter-at-a-time logic, we test whether the trend continues at 1.0. This is a deliberate probe of the parameter's optimum: if 1.0 is better we keep climbing, and if it is worse we have bracketed the optimum around 0.8.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8m.pt")

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,
    patience=30,
    cache="ram",
    device=0,
    amp=True,
    seed=0,
    copy_paste=0.5,   # unchanged - helps rare-class recall
    cls=1.0            # up from 0.8 - cls=0.8 reduced people_wheelchair -> wheelchair confusion from 26% to 12%, testing if pushing further helps more
)

### Visualize Training Results (Attempt 4)
This cell plots the metrics and confusion matrix for the `cls=1.0` run.

**Why this step:** This is the verdict cell for the `cls` sweep — and the answer is clear: **`cls=1.0` performed *worse* than `cls=0.8`**. Over-weighting classification starts to hurt the other loss components (box regression), so the optimum sits at ≈0.8. Conclusion: `cls` is a *maxed-out* lever within this sweep, just like `copy_paste`, making **train-4 (`cls=0.8`, `copy_paste=0.5`) the best run *so far*** — but not the final model: more experiments follow (freeze/unfreeze, explicit SGD, cosine LR, extra augmentation), and the dedicated comparison cell near the end evaluates every run on the `valid` split before anything is selected or exported.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Update the path if your run was saved elsewhere (check the "save_dir" printed above)
run_dir = "/content/runs/detect/train-5"

# Show results.png (loss, mAP, precision/recall curves over epochs)
img_results = mpimg.imread(f"{run_dir}/results.png")
plt.figure(figsize=(16, 10))
plt.imshow(img_results)
plt.axis("off")
plt.title("Training Results")
plt.show()

# Show confusion_matrix.png (normalized version - easier to read as percentages)
img_cm = mpimg.imread(f"{run_dir}/confusion_matrix_normalized.png")
plt.figure(figsize=(10, 8))
plt.imshow(img_cm)
plt.axis("off")
plt.title("Confusion Matrix (Normalized)")
plt.show()

### Evaluate train-4 on the Validation Split (Numeric + Visual)
This cell loads the weights from the `cls=0.8` run (`train-4/weights/best.pt`) and runs `model.val()` for numeric metrics (mAP50, mAP50-95, precision, recall) on the `valid` split, then `model.predict()` with `save=True` to produce annotated images. Inference uses `conf=0.15` (low, so no real case is missed — appropriate for a safety/accessibility use-case) and `iou=0.6`. The `test` split is not touched by this cell, by design — it stays unseen by every model-selection decision made up to this point, and is used exactly once, later, in the dedicated "Final Held-Out Check on the Test Split" cell near the end of the notebook.

**Why this step:** train-4 is the strongest run among the four `copy_paste`/`cls` sweep attempts, so it's worth a dedicated evaluation here. This is **not** the final model-selection step, though — several more architectures/optimizers are tried further down (freeze/unfreeze, explicit SGD, cosine LR, extra augmentation), and the dedicated comparison cell near the end of the notebook evaluates **every** run on this same `valid` split before anything is exported. Treat the numbers below as one data point among several, not a conclusion.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import yaml

# --- Paths ---
# Deliberately using the "valid" split only. The "test" split is never
# referenced anywhere in this notebook, so it stays fully unseen by every
# model and every model-selection decision made here.
valid_images_path_actual = Path(dataset.location) / "valid" / "images"
data_yaml_path = Path(dataset.location) / "data.yaml"
best_weights = Path("/content/runs/detect/train-4/weights/best.pt")

with open(data_yaml_path) as f:
    data_cfg = yaml.safe_load(f)

if not best_weights.exists():
    raise FileNotFoundError(
        f"Weights not found at {best_weights}. Finish the train-4 (Attempt 3) cell first."
    )

# --- Step 1: Load the trained model ---
model = YOLO(str(best_weights))

# --- Step 2: Quantitative evaluation on the valid split ---
metrics = model.val(
    data=str(data_yaml_path),
    split="val", # Changed from "valid" to "val" to match data.yaml
    conf=0.15,
    iou=0.6,
    name="valid_eval_train4"
)

print("\n=== Metrics (valid split) ===")
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

# --- Step 3: Visual run ---
results = model.predict(
    source=str(valid_images_path_actual),
    conf=0.15,
    iou=0.6,
    save=True,
    save_txt=True,
    name="valid_predict_train4"
)

print(f"\nVisual results saved to: runs/detect/valid_predict_train4")

### Error Analysis: Per-Image Count Mismatches
This cell compares, for every valid-split image, the number of predicted `person` instances against the ground-truth count, then ranks images by the largest over- and under-predictions and visualizes the worst offenders.

**Why this step:** The evaluation above told us *that* `person` is the weak class but not *why*. Before touching any more hyperparameters (which the previous experiments showed are exhausted), we look at the actual failure cases. The visualization is revealing: the worst over-predictions are all **dense crowd scenes** (stadiums, parades, parks) where the model detects far more people than the ground truth lists — the model appears to be finding *real* people that the annotators never labeled. That hypothesis needs statistical confirmation, which the next cell provides.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

model = YOLO("runs/detect/train-4/weights/best.pt")

# Deliberately using "valid", not "test" - see the notes earlier in the
# notebook about keeping the test split fully unseen.
valid_images = Path(dataset.location) / "valid" / "images"
valid_labels = Path(dataset.location) / "valid" / "labels"

# Run predictions on the valid split with the same settings
results = model.predict(
    source=str(valid_images),
    conf=0.15,
    iou=0.6,
    save=False,
    verbose=False
)

# For each image, count predicted persons vs ground-truth persons
# person class id - check your data.yaml mapping (usually 1)
person_id = data_cfg["names"].index("person")

mismatches = []
for r in results:
    img_path = Path(r.path)
    label_path = valid_labels / (img_path.stem + ".txt")

    # ground truth count
    gt_person_count = 0
    if label_path.exists():
        with open(label_path) as f:
            for line in f:
                cls_id = int(line.split()[0])
                if cls_id == person_id:
                    gt_person_count += 1

    # predicted count
    pred_person_count = sum(1 for c in r.boxes.cls if int(c) == person_id)

    diff = pred_person_count - gt_person_count
    mismatches.append((img_path, gt_person_count, pred_person_count, diff))

# Sort by biggest over-prediction (most false positives)
mismatches.sort(key=lambda x: x[3], reverse=True)

print("Top 10 images where model over-predicts 'person' (potential false positives):")
for img_path, gt, pred, diff in mismatches[:10]:
    print(f"{img_path.name}: GT={gt}, Predicted={pred}, Diff={diff:+d}")

print("\nTop 10 images where model under-predicts 'person' (potential missed detections):")
for img_path, gt, pred, diff in mismatches[-10:]:
    print(f"{img_path.name}: GT={gt}, Predicted={pred}, Diff={diff:+d}")

# Visualize the top 4 over-prediction cases
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (img_path, gt, pred, diff) in zip(axes.flat, mismatches[:4]):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"{img_path.name}\nGT person={gt}, Predicted={pred} (diff={diff:+d})", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


### Correlation Analysis: Crowd Size vs. Prediction Error
This cell computes the correlation between ground-truth crowd size and the prediction difference, and breaks the error down by density bins.

**Why this step:** The visual inspection suggested a pattern (crowds → missing labels), but four images are an anecdote, not evidence. This cell quantifies it with the actual printed correlation coefficient and per-density-bin averages below — read those numbers directly from the cell output rather than from this note, since earlier drafts of this notebook quoted a specific correlation value and per-bin averages here that did not match what the code actually printed. Directionally, a positive correlation here would support the interpretation that the model's measured weakness on `person` is driven at least in part by **incomplete ground-truth annotations in crowded images** — a data-quality problem, not purely a model problem — but treat any specific percentage as provisional until very few images populate the high-density bins.

In [ ]:
import numpy as np

gt_counts = np.array([m[1] for m in mismatches])
diffs = np.array([m[3] for m in mismatches])

corr = np.corrcoef(gt_counts, diffs)[0, 1]
print(f"Correlation between crowd size and prediction diff: {corr:.3f}")

# Division by density
bins = [(0, 5), (5, 15), (15, 30), (30, 1000)]
for lo, hi in bins:
    mask = (gt_counts >= lo) & (gt_counts < hi)
    if mask.sum() > 0:
        avg_diff = diffs[mask].mean()
        print(f"GT person in [{lo},{hi}): {mask.sum()} images, avg diff = {avg_diff:+.2f}")

### Identify Extreme Crowd Images
Note: the code cell below does not isolate or visualize individual images — it only prints a decision to skip crowd-based filtering (see the next markdown cell for why). If you want to actually inspect the specific high-crowd images, re-use the ranking/visualization logic from the "Error Analysis: Per-Image Count Mismatches" cell above, filtered to `gt_counts >= 30`.

**Why this step:** The correlation analysis above showed a positive correlation between crowd size and prediction error, concentrated in a small number of high-density images — read the exact counts and per-bin averages from that cell's own printed output rather than from a fixed number here, since earlier drafts of this notebook quoted specific counts that did not match what the code actually printed. Since their ground truth is plausibly incomplete in dense scenes, evaluating the model against them can punish correct detections. We identify them precisely so the next cell can measure the model *without* this label noise.

In [ ]:
print(
    "Skipping crowd-based filtering.\n"
    "Continue evaluation on the full `valid` split "
    "(no custom test_clean_filtered, and the `test` split is not used here)."
)


### Filter and Re-evaluate (Skipped)
Older runs built `test_clean_filtered` by dropping extreme crowd images. That step is **skipped** here: we keep the full `valid` split for all metrics so results stay comparable across hyperparameter experiments. (Filtering `test` is not relevant either, since this notebook does not evaluate on `test` until the single held-out check near the end.)

In [ ]:
print(
    "Skipping re-evaluation on a filtered set.\n"
    "All further evaluations use the original `valid` split "
    "(the `test` split is not used again until the single held-out check near the end of this notebook)."
)


### Analyze Overlapping Bounding Boxes (Double Detections)
This cell computes IoU between every pair of predicted boxes in each valid-split image and flags pairs with IoU > 0.5, broken down by class pair.

**Why this step:** Our diagnostic decision tree has one branch we have not yet tested: NMS/IoU issues — the model predicting both `person` and `people_wheelchair` on the same object. Since this check requires only inference (no retraining), it is cheap to close this gap before concluding the analysis. Read the printed class-pair breakdown below for the actual counts — earlier drafts of this notebook stated specific totals here that did not match the cell's own printed output, so do not rely on any fixed number until you see it printed. The `person`↔`people_wheelchair` pairs are the specific confusion we care about; worth seeing with our own eyes regardless of the exact count.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import numpy as np

model = YOLO("runs/detect/train-4/weights/best.pt")

# Deliberately using "valid", not "test" - the test split is kept fully
# unseen throughout this notebook.
valid_images_path = Path(dataset.location) / "valid" / "images"

results = model.predict(
    source=str(valid_images_path),
    conf=0.15,
    iou=0.6,
    save=False,
    verbose=False
)

def box_iou(box1, box2):
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

names = data_cfg["names"]
overlap_cases = []

for r in results:
    boxes = r.boxes.xyxy.cpu().numpy()
    classes = r.boxes.cls.cpu().numpy().astype(int)

    n = len(boxes)
    for i in range(n):
        for j in range(i+1, n):
            iou = box_iou(boxes[i], boxes[j])
            if iou > 0.5:  # significant overlap between two separate detections
                overlap_cases.append((
                    Path(r.path).name,
                    names[classes[i]], names[classes[j]],
                    round(iou, 2)
                ))

print(f"Found {len(overlap_cases)} overlapping detection pairs (IoU > 0.5) across {len(results)} images")

# Breakdown by class pair
from collections import Counter
pair_counts = Counter(tuple(sorted((c1, c2))) for _, c1, c2, _ in overlap_cases)
print("\nClass-pair breakdown:")
for (c1, c2), cnt in pair_counts.most_common():
    print(f"  {c1} <-> {c2}: {cnt} overlapping pairs")

# Show the worst examples
print("\nTop 10 highest-IoU overlapping pairs (potential double-detections/confusion):")
for name, c1, c2, iou in sorted(overlap_cases, key=lambda x: -x[3])[:10]:
    print(f"  {name}: {c1} <-> {c2}, IoU={iou}")


### Visualize Overlapping Detections
This cell draws both boxes for each flagged `person`↔`people_wheelchair` overlap case (the exact count is printed by the previous cell) so they can be inspected visually.

**Why this step:** A handful of flagged cases could mean a systematic model failure or genuinely ambiguous images — only looking at them can tell. Check the confidence scores annotated on each image below: if most involve low, comparable confidence on both candidate labels, that supports the model *hesitating between two plausible readings* rather than systematically failing. Combined with the earlier findings, this closes the error analysis for the model evaluated so far. Two more techniques remain untested, so we turn to them next.

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

model = YOLO("runs/detect/train-4/weights/best.pt")
# Deliberately using "valid", not "test" - the test split is kept fully unseen.
valid_images_path = Path(dataset.location) / "valid" / "images"
names = data_cfg["names"]

results = model.predict(
    source=str(valid_images_path),
    conf=0.15,
    iou=0.6,
    save=False,
    verbose=False,
)


def box_iou(box1, box2):
  x1 = max(box1[0], box2[0])
  y1 = max(box1[1], box2[1])
  x2 = min(box1[2], box2[2])
  y2 = min(box1[3], box2[3])
  inter = max(0, x2 - x1) * max(0, y2 - y1)
  area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
  area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
  union = area1 + area2 - inter
  return inter / union if union > 0 else 0


flagged = []
for r in results:
  boxes = r.boxes.xyxy.cpu().numpy()
  classes = r.boxes.cls.cpu().numpy().astype(int)
  confs = r.boxes.conf.cpu().numpy()
  n = len(boxes)
  for i in range(n):
    for j in range(i + 1, n):
      c1_name, c2_name = names[classes[i]], names[classes[j]]
      if {c1_name, c2_name} == {"person", "people_wheelchair"}:
        iou = box_iou(boxes[i], boxes[j])
        if iou > 0.5:
          flagged.append((
              r.path,
              boxes[i],
              classes[i],
              confs[i],
              boxes[j],
              classes[j],
              confs[j],
              iou,
          ))

print(f"Found {len(flagged)} person<->people_wheelchair overlap cases")

colors_dict = {
    "person": (255, 51, 51),
    "people_wheelchair": (51, 255, 51),
    "wheelchair": (51, 153, 255),
}

n_show = len(flagged)
cols = 3
rows = (n_show + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
axes = np.array(axes).reshape(-1)


# Helper function to draw text with a black background to avoid overlap
def draw_text_with_bg(
    img, text, pt, color, thickness=2, font_scale=0.6, bg_color=(0, 0, 0)
):
  font = cv2.FONT_HERSHEY_SIMPLEX
  (text_w, text_h), baseline = cv2.getTextSize(
      text, font, font_scale, thickness
  )
  x, y = pt
  cv2.rectangle(
      img, (x, y - text_h - 4), (x + text_w + 4, y + baseline), bg_color, -1
  )
  cv2.putText(img, text, (x + 2, y - 2), font, font_scale, color, thickness)


for idx, (path, box1, cls1, conf1, box2, cls2, conf2, iou) in enumerate(
    flagged
):
  img = cv2.imread(str(path))
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

  # Determine the winner (higher confidence)
  if conf1 >= conf2:
    win_box, win_cls, win_conf = box1, cls1, conf1
    lose_box, lose_cls, lose_conf = box2, cls2, conf2
  else:
    win_box, win_cls, win_conf = box2, cls2, conf2
    lose_box, lose_cls, lose_conf = box1, cls1, conf1

  # 1. Draw the losing prediction (thin line, text at the bottom of the box)
  lx1, ly1, lx2, ly2 = lose_box.astype(int)
  l_cname = names[lose_cls]
  l_color = colors_dict.get(l_cname, (200, 200, 200))
  cv2.rectangle(img, (lx1, ly1), (lx2, ly2), l_color, 2)
  draw_text_with_bg(
      img,
      f"{l_cname} {lose_conf:.2f}",
      (lx1, min(ly2 - 10, img.shape[0] - 10)),
      l_color,
      thickness=1,
  )

  # 2. Draw the winning prediction (thick line, text at the top of the box)
  wx1, wy1, wx2, wy2 = win_box.astype(int)
  w_cname = names[win_cls]
  w_color = colors_dict.get(w_cname, (0, 255, 0))
  cv2.rectangle(img, (wx1, wy1), (wx2, wy2), w_color, 4)
  draw_text_with_bg(
      img,
      f"WINNER: {w_cname} {win_conf:.2f}",
      (wx1, max(wy1 - 8, 15)),
      (0, 255, 0),
      thickness=2,
  )

  ax = axes[idx]
  ax.imshow(img)
  ax.set_title(
      f"{Path(path).name}\nIoU={iou:.2f} | Winner: {w_cname}"
      f" ({win_conf:.2f} vs {lose_conf:.2f})",
      fontsize=10,
      fontweight="bold",
  )
  ax.axis("off")

for idx in range(n_show, len(axes)):
  axes[idx].axis("off")

plt.tight_layout()
plt.show()

### Transfer-Learning Experiment: Freeze / Unfreeze Backbone
This cell implements a two-phase strategy: **Phase A** freezes the 10 backbone layers (`freeze=10`) and trains only the detection head for a short warm-up; **Phase B** loads that checkpoint, unfreezes everything, and continues full fine-tuning with a lower learning rate.

**Why this step:** Freeze/unfreeze is the classic transfer-learning recipe and a required technique to demonstrate — and it is also a fair scientific question: could staged fine-tuning beat direct fine-tuning? Compare this run's final validation metrics (printed below) against the other runs in the comparison table generated later in the notebook rather than assuming the outcome in advance — an earlier draft of this notebook claimed a specific `person` recall value here that did not match this run's own printed output. A side discovery from these logs proved valuable regardless of the final numbers: `optimizer=auto` **silently ignores any `lr0` you pass** — meaning learning rate was never actually varied in the `copy_paste`/`cls` sweep above. That motivates the next experiment, which sets the optimizer explicitly.

In [ ]:
from ultralytics import YOLO

# ============================================================
# Phase A: Freeze — Short training with frozen backbone
# ============================================================
# yolov8m has 10 backbone layers (indices 0-9) followed by the head.
# freeze=10 freezes layers 0-9 (the full backbone), leaving only the
# detection head trainable. This trains fast and adapts the head to
# our 3 classes without disturbing the pretrained visual features.

model = YOLO("yolov8m.pt")

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,          # Short - just to warm up the head
    patience=10,
    cache="ram",
    device=0,
    amp=True,
    seed=0,
    freeze=10,           # Freeze full backbone (layers 0-9)
    lr0=0.001,           # Regular LR, the head starts relatively from zero
    name="train-freeze"
)

# ============================================================
# Phase B: Unfreeze — Continue from checkpoint, all layers open
# ============================================================
# Load the weights that have already been "warmed up" in phase A, and continue to train
# the entire network (backbone + head) with a lower LR to avoid
# destroying the pre-trained weights all at once.

model_unfrozen = YOLO("runs/detect/train-freeze/weights/best.pt")

model_unfrozen.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,
    patience=30,
    cache="ram",
    device=0,
    amp=True,
    seed=0,
    freeze=None,          # Do not freeze anything - full fine-tuning
    lr0=0.0005,            # Lower LR than phase A - gentle updates
    copy_paste=0.5,        # Preserve what already worked well for us
    cls=0.8,
    name="train-unfreeze-full"
)

### Training Experiment: Explicit SGD with `lr0=0.001`
This cell trains with `optimizer="SGD"` set explicitly and `lr0=0.001` (10× lower than the default 0.01), keeping the proven `copy_paste=0.5` and `cls=0.8` unchanged.

**Why this step:** The freeze experiment's logs revealed that every previous run used `optimizer=auto`, which overrides any `lr0` we pass — so learning rate, arguably the most influential hyperparameter, was never genuinely tested. Setting the optimizer explicitly is the only way to make `lr0` take effect. We choose 0.001 because fine-tuning pretrained weights conventionally favors a lower rate than training from scratch. As always, this changes effectively one factor relative to train-4.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8m.pt")

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,
    patience=30,
    cache="ram",
    device=0,
    amp=True,
    seed=0,
    copy_paste=0.5,
    cls=0.8,
    optimizer="SGD",   # Critical - without this, lr0 is ignored
    lr0=0.001,
    name="train-lr-experiment"
)

### Evaluate the SGD Experiment on the Validation Split
This cell evaluates the SGD + `lr0=0.001` run on the same `valid` split used for train-4, so the comparison is fair. The `test` split is not touched here — it is reserved for the single held-out check near the end of the notebook.

**Why this step:** Compare the SGD challenger to train-4 on the identical `valid` split before deciding which model to keep, without ever touching the held-out `test` split.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import yaml

data_root = Path(dataset.location)
data_yaml_path = data_root / "data.yaml"

sgd_weights = Path("runs/detect/train-lr-experiment/weights/best.pt")
if not sgd_weights.exists():
    raise FileNotFoundError(
        f"Weights not found at {sgd_weights}. Run the SGD training experiment cell first."
    )

model_new = YOLO(str(sgd_weights))

# Deliberately evaluating on "val" (as defined in data.yaml), not "test".
# The test split is kept fully unseen throughout this notebook.
metrics_new = model_new.val(
    data=str(data_yaml_path),
    split="val",
    conf=0.15,
    iou=0.6,
    name="valid_eval_lr_experiment"
)

print("=== train-lr-experiment on valid split ===")
print("mAP50:", metrics_new.box.map50)
print("mAP50-95:", metrics_new.box.map)
print("Precision:", metrics_new.box.mp)
print("Recall:", metrics_new.box.mr)

### Train YOLOv8m - Experiment 5 (Learning Rate Optimization and Cosine Scheduler)
This cell continues training with a new learning rate ($lr_0=0.0005$) and adds a Cosine Learning Rate Scheduler. The combination of a lower learning rate and a dynamic learning rate schedule can help the model converge more smoothly and efficiently, especially in the later stages of training. All other parameters remain as set in previous experiments.

**Why this step:**
*   **Learning Rate ($lr_0=0.0005$):** Experiment 4 showed that a learning rate of 0.001 worked well. Testing 0.0005 will help determine if a lower rate can further improve performance, as fine-tuning usually benefits from starting with a lower learning rate.
*   **Cosine Learning Rate Scheduler ($cos_lr=True$):** The Cosine Annealing Scheduler gradually changes the learning rate during training. It starts at the initial learning rate ($lr_0$) and slowly reduces it towards zero following a cosine function. This allows the model to take 'larger steps' early in training to explore the parameter space, and 'smaller steps' towards the end of training to fine-tune the weights and avoid overshooting minima.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,
    patience=30,
    cache="ram",
    device=0,
    amp=True,
    seed=0,
    copy_paste=0.5,
    cls=0.8,
    optimizer="SGD",
    lr0=0.0005,  # Trying a new, lower initial learning rate
    cos_lr=True, # Enable Cosine learning rate scheduler
    mixup=0.1,
    scale=0.7,
    hsv_h=0.03,
    hsv_s=0.9,
    hsv_v=0.6,
    name="train-lr-cosine-experiment"
)

### Train YOLOv8m — Attempt 6 (Enhanced Augmentations for Recall)
This cell further enhances the training process by incorporating more aggressive data augmentation techniques: `mixup`, wider `scale` variations, and increased `hsv_h`, `hsv_s`, `hsv_v` values. We retain `copy_paste=0.5`, `cls=0.8`, `optimizer="SGD"`, and `lr0=0.001` from the earlier explicit-SGD run (note: the cosine-LR run scored slightly higher on valid, so this choice of `lr0=0.001` is a judgment call, not strictly "the best so far" — the final comparison cell settles the question empirically).

**Why this step:** The goal is to address the remaining recall issues, particularly for the 'person' class, by forcing the model to learn from more diverse and 'harder' examples. `Mixup` helps the model generalize by creating interpolated images, `scale` variations enable detection of objects at various distances, and `HSV` adjustments improve robustness to different lighting and color conditions. This should make the model more resilient to real-world variability and improve its ability to detect previously missed instances.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,
    patience=30,
    cache="ram",
    device=0,
    amp=True,
    seed=0,
    copy_paste=0.5,
    cls=0.8,
    optimizer="SGD",
    lr0=0.001,
    mixup=0.1,  # Increased from 0.0 for more diverse training examples
    scale=0.7,  # Wider scaling range for detecting objects at various distances
    hsv_h=0.03, # Increased hue augmentation
    hsv_s=0.9,  # Increased saturation augmentation
    hsv_v=0.6,  # Increased brightness augmentation
    name="train-augmented-lr"
)

### Compare All Training Runs and Select the Best Model
This section does **not** assume any single run (train-4, the SGD run, the freeze/unfreeze run, the cosine-LR run, or the augmented run) is the best model. Instead it evaluates **every** run that produced a `best.pt` under `runs/detect/` on the `valid` split, with the exact same inference settings (`conf=0.15`, `iou=0.6`), and builds one comparison table. The run selected as "best" below is chosen automatically from that table — nothing here is hard-coded.

**The `test` split is not loaded or evaluated in this comparison cell.** Model selection is based on `valid` only, precisely so that the choice of "best model" is not itself informed by the one split meant to stay held out through selection. The dedicated "Final Held-Out Check on the Test Split" cell near the end of the notebook evaluates only the already-selected model on `test`, exactly once — `test` is never fed back into a decision about which run to keep.

**Why this step:** Earlier versions of this notebook picked train-4 as the final model by narrative ("the hyperparameter search converged on train-4") without ever comparing it, on the same footing, against runs that were tried later (SGD, freeze/unfreeze, cosine LR, extra augmentation). Some of those later runs may score higher. This cell removes that gap: it is the single source of truth for which run is exported at the end of the notebook, and it does so without touching the held-out test data.

In [ ]:
import glob
from pathlib import Path
import pandas as pd
from ultralytics import YOLO
import yaml

# NOTE: this cell evaluates every run on the "val" split only.
# The "test" split is never referenced here or anywhere else in this
# notebook, so model selection cannot be biased by it.
data_root = Path(dataset.location)
data_yaml_path = data_root / "data.yaml"

# --- Discover every run that produced trained weights ---
weight_paths = sorted(glob.glob("runs/detect/*/weights/best.pt"))
# De-duplicate re-validation runs like "valid_eval*" / "valid_predict*" that are not training runs
weight_paths = [
    p for p in weight_paths
    if not any(skip in p for skip in ["valid_eval", "valid_predict", "compare_", "test_val", "test_predict"])
]

print(f"Found {len(weight_paths)} trained run(s) to evaluate:")
for p in weight_paths:
    print(" -", p)

rows = []
for wp in weight_paths:
    run_name = Path(wp).parent.parent.name
    try:
        model = YOLO(wp)
        metrics = model.val(
            data=str(data_yaml_path),
            split="val", # Changed from "valid" to "val" to match data.yaml structure
            conf=0.15,
            iou=0.6,
            name=f"compare_{run_name}",
            verbose=False,
        )
        rows.append({
            "run": run_name,
            "weights": wp,
            "mAP50-95": metrics.box.map,
            "mAP50": metrics.box.map50,
            "precision": metrics.box.mp,
            "recall": metrics.box.mr,
        })
        print(f"  {run_name}: mAP50-95={metrics.box.map:.4f}  mAP50={metrics.box.map50:.4f}  "
              f"P={metrics.box.mp:.4f}  R={metrics.box.mr:.4f}")
    except Exception as e:
        print(f"  {run_name}: evaluation failed ({e})")

if len(rows) == 0:
    raise RuntimeError("No trained runs were successfully evaluated. Check dataset paths and YAML keys.")

comparison_df = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False).reset_index(drop=True)

print("\n=== Comparison table on the VALID split (sorted by mAP50-95, best first) ===")
display(comparison_df)

BEST_RUN = comparison_df.iloc[0]["run"]
BEST_WEIGHTS = Path(comparison_df.iloc[0]["weights"])

print(f"\nSelected best run by mAP50-95 (on valid): '{BEST_RUN}'")
print(f"Weights: {BEST_WEIGHTS}")

### Visualize All Failure Cases of the Selected Best Model
Using `BEST_WEIGHTS` from the comparison above (not a hard-coded run), this section re-runs the same two error analyses performed earlier in the notebook — person-count mismatches and person↔people_wheelchair overlapping detections — but on the actual best-performing run, and displays **every** flagged image rather than only a top-10 sample. This runs on the `valid` split, consistent with the comparison cell above — the `test` split is not touched here either.

**Why this step:** The earlier error-analysis cells were run against train-4 specifically, before the best run was known. Re-running them here against the actual best model keeps the error analysis consistent with whichever run ends up selected, and showing every case (not just the worst few) gives a complete picture for a manual data-quality pass — all on `valid`, keeping `test` unexposed until the dedicated held-out check after export.

In [ ]:
import math
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

best_model = YOLO(str(BEST_WEIGHTS))

# Deliberately evaluating on "valid" - the "test" split is not used
# anywhere in this notebook, including here.

valid_images_dir = Path(dataset.location) / "valid" / "images"
valid_labels_dir = Path(dataset.location) / "valid" / "labels"
names = data_cfg["names"]
person_id = names.index("person")

results_all = best_model.predict(
    source=str(valid_images_dir),
    conf=0.15,
    iou=0.6,
    save=False,
    verbose=False,
)


def box_iou(box1, box2):
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0


def draw_text_with_bg(img, text, pt, color, thickness=2, font_scale=0.6, bg_color=(0, 0, 0)):
    font = cv2.FONT_HERSHEY_SIMPLEX
    (text_w, text_h), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    x, y = pt
    cv2.rectangle(img, (x, y - text_h - 4), (x + text_w + 4, y + baseline), bg_color, -1)
    cv2.putText(img, text, (x + 2, y - 2), font, font_scale, color, thickness)


# --- Pass 1: person-count mismatches (every image, not just top-10) ---
count_mismatches = []
overlap_cases = []

for r in results_all:
    img_path = Path(r.path)
    label_path = valid_labels_dir / (img_path.stem + ".txt")

    gt_person_count = 0
    if label_path.exists():
        with open(label_path) as f:
            for line in f:
                if line.strip() and int(line.split()[0]) == person_id:
                    gt_person_count += 1

    pred_person_count = sum(1 for c in r.boxes.cls if int(c) == person_id)
    diff = pred_person_count - gt_person_count
    if diff != 0:
        count_mismatches.append((img_path, gt_person_count, pred_person_count, diff))

    boxes = r.boxes.xyxy.cpu().numpy()
    classes = r.boxes.cls.cpu().numpy().astype(int)
    confs = r.boxes.conf.cpu().numpy()
    n = len(boxes)
    for i in range(n):
        for j in range(i + 1, n):
            c1n, c2n = names[classes[i]], names[classes[j]]
            if {c1n, c2n} == {"person", "people_wheelchair"}:
                iou = box_iou(boxes[i], boxes[j])
                if iou > 0.5:
                    overlap_cases.append((img_path, boxes[i], classes[i], confs[i],
                                           boxes[j], classes[j], confs[j], iou))

count_mismatches.sort(key=lambda x: abs(x[3]), reverse=True)
print(f"Total images with a person-count mismatch: {len(count_mismatches)} / {len(results_all)}")
print(f"Total person<->people_wheelchair overlap cases (IoU > 0.5): {len(overlap_cases)}")

# --- Plot every count-mismatch case, GT (green) vs Predicted (red) ---
if count_mismatches:
    n_show = len(count_mismatches)
    cols = 3
    rows = math.ceil(n_show / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
    axes = np.array(axes).reshape(-1)

    for idx, (img_path, gt, pred, diff) in enumerate(count_mismatches):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        label_path = valid_labels_dir / (img_path.stem + ".txt")
        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if parts and int(parts[0]) == person_id:
                        _, xc, yc, bw, bh = map(float, parts)
                        x1 = int((xc - bw / 2) * w); y1 = int((yc - bh / 2) * h)
                        x2 = int((xc + bw / 2) * w); y2 = int((yc + bh / 2) * h)
                        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

        r = next(rr for rr in results_all if Path(rr.path) == img_path)
        for b in r.boxes:
            if int(b.cls) == person_id:
                bx = b.xyxy[0].cpu().numpy().astype(int)
                cv2.rectangle(img, (bx[0], bx[1]), (bx[2], bx[3]), (255, 0, 0), 2)

        axes[idx].imshow(img)
        axes[idx].set_title(f"{img_path.name}\nGT={gt}  Pred={pred}  diff={diff:+d}", fontsize=9)
        axes[idx].axis("off")

    for idx in range(n_show, len(axes)):
        axes[idx].axis("off")
    plt.suptitle("All person-count mismatches — green=Ground Truth, red=Predicted", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No person-count mismatches found for this model.")

# --- Plot every overlap case, winner vs loser by confidence ---
colors_dict = {"person": (255, 51, 51), "people_wheelchair": (51, 255, 51), "wheelchair": (51, 153, 255)}

if overlap_cases:
    n_show = len(overlap_cases)
    cols = 3
    rows = math.ceil(n_show / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
    axes = np.array(axes).reshape(-1)

    for idx, (img_path, box1, cls1, conf1, box2, cls2, conf2, iou) in enumerate(overlap_cases):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if conf1 >= conf2:
            win_box, win_cls, win_conf = box1, cls1, conf1
            lose_box, lose_cls, lose_conf = box2, cls2, conf2
        else:
            win_box, win_cls, win_conf = box2, cls2, conf2
            lose_box, lose_cls, lose_conf = box1, cls1, conf1

        lx1, ly1, lx2, ly2 = lose_box.astype(int)
        l_cname = names[lose_cls]
        cv2.rectangle(img, (lx1, ly1), (lx2, ly2), colors_dict.get(l_cname, (200, 200, 200)), 2)
        draw_text_with_bg(img, f"{l_cname} {lose_conf:.2f}", (lx1, min(ly2 - 10, img.shape[0] - 10)),
                           colors_dict.get(l_cname, (200, 200, 200)), thickness=1)

        wx1, wy1, wx2, wy2 = win_box.astype(int)
        w_cname = names[win_cls]
        cv2.rectangle(img, (wx1, wy1), (wx2, wy2), colors_dict.get(w_cname, (0, 255, 0)), 4)
        draw_text_with_bg(img, f"WINNER: {w_cname} {win_conf:.2f}", (wx1, max(wy1 - 8, 15)),
                           (0, 255, 0), thickness=2)

        axes[idx].imshow(img)
        axes[idx].set_title(f"{img_path.name}\nIoU={iou:.2f} | winner: {w_cname} ({win_conf:.2f} vs {lose_conf:.2f})",
                             fontsize=9, fontweight="bold")
        axes[idx].axis("off")

    for idx in range(n_show, len(axes)):
        axes[idx].axis("off")
    plt.suptitle("All person<->people_wheelchair overlap cases", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No person<->people_wheelchair overlap cases found for this model.")


### Export Final Weights for Documentation & Handoff
This cell copies the weights of whichever run was selected as best in the comparison cell above (`BEST_WEIGHTS`) to a clearly named file, prints a **model card** (architecture, classes, selected run, actual test metrics, dataset version, recommended inference settings), and downloads the file from Colab. An optional commented block also saves a copy to Google Drive.

**Why this step:** Colab's filesystem is ephemeral. Exporting `best.pt` under a descriptive name with a model card makes the run reproducible without re-reading the whole notebook. Because the exported filename and model card are generated from the comparison results rather than typed in by hand, this cell can be re-run unchanged if a future training experiment turns out to beat everything evaluated so far.

In [ ]:
import shutil
from pathlib import Path
from datetime import datetime

# --- Final weights export for documentation / handoff ---
# BEST_WEIGHTS / BEST_RUN / comparison_df come from the comparison cell above,
# which evaluated every run on the "valid" split only. Nothing here is
# hard-coded to a specific run name, and the "test" split has not been
# touched by training, by any experiment, or by this selection.
assert "BEST_WEIGHTS" in dir(), "Run the comparison/evaluation cell above first."
FINAL_WEIGHTS = Path(BEST_WEIGHTS)
EXPORT_NAME = f"wheelchair_yolov8m_final_{BEST_RUN}.pt"

assert FINAL_WEIGHTS.exists(), f"Weights not found at {FINAL_WEIGHTS} - check the run directory name"

shutil.copy2(FINAL_WEIGHTS, EXPORT_NAME)
size_mb = Path(EXPORT_NAME).stat().st_size / 1e6
print(f"Exported: {EXPORT_NAME} ({size_mb:.1f} MB)")
print(f"Source run: {BEST_RUN}")
print(f"Export date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

best_row = comparison_df.iloc[0]

# Model card for the documentation owner - metrics and run name are pulled
# directly from the comparison table, not typed in by hand.
print(f"""
=== MODEL CARD ===
Architecture : YOLOv8m
Task         : Object detection, 3 classes
Classes      : people_wheelchair (0), person (1), wheelchair (2)
Selected run : {BEST_RUN}  (chosen automatically: highest mAP50-95 among all evaluated runs, on "valid")
Valid metrics: mAP50-95={best_row['mAP50-95']:.4f}  mAP50={best_row['mAP50']:.4f}  precision={best_row['precision']:.4f}  recall={best_row['recall']:.4f}
Dataset      : Roboflow first-group-project / wheelchair-9qvfx-bchvo, version at {dataset.location} (see the "Download Dataset" cell for the exact version actually downloaded)
Splits       : train / valid / test from the downloaded dataset at {dataset.location}
Test set     : Held out through every training run, hyperparameter decision, and
               model-selection step above. Evaluated exactly once, after this export,
               in the "Final Held-Out Check on the Test Split" cell below.
Inference    : conf=0.15, iou=0.6 (safety-first: minimize missed cases)
Notes        : See the full comparison table earlier in the notebook (variable `comparison_df`)
               for every run's metrics side by side, all computed on "valid". Every training
               run in this notebook requests epochs=30 with early stopping via patience
               (check each run's own training cell for its exact settings).
""")

# Option A: direct download from Colab
from google.colab import files
files.download(EXPORT_NAME)

# Option B (alternative): persistent backup to Google Drive - uncomment to use
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy2(EXPORT_NAME, f"/content/drive/MyDrive/{EXPORT_NAME}")
# print("Also copied to Google Drive: MyDrive/" + EXPORT_NAME)


### Final Held-Out Check on the Test Split (Run Once)
This is the only cell in the entire notebook that touches the `test` split.
It loads `BEST_WEIGHTS` — the run already selected by the comparison cell above,
based purely on `valid` performance — and evaluates it once on `test`, using
the same `conf=0.15`, `iou=0.6` settings used everywhere else, so the number
is comparable to the `valid` metrics already reported.

**Why this step:** Every hyperparameter decision, every run comparison, and
the final model selection all happened using only `train` and `valid`. `test`
was deliberately left untouched throughout so it could serve as one genuinely
independent check — run *after* the model was already locked in, not as part
of choosing it. This cell is that check. Do not re-run earlier cells and come
back to this one with a different `BEST_WEIGHTS` — that would defeat the
purpose of keeping `test` held out.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

assert "BEST_WEIGHTS" in dir(), "Run the comparison cell above first to select BEST_WEIGHTS."

data_yaml_path = Path(dataset.location) / "data.yaml"

print(f"Final held-out evaluation on TEST split")
print(f"Model: {BEST_WEIGHTS}")
print("(This is the only cell in this notebook that uses the test split.)\n")

model = YOLO(str(BEST_WEIGHTS))

test_metrics = model.val(
    data=str(data_yaml_path),
    split="test",
    conf=0.15,
    iou=0.6,
    name="final_test_eval",
)

print("\n=== Final metrics (TEST split, held-out) ===")
print("mAP50:", test_metrics.box.map50)
print("mAP50-95:", test_metrics.box.map)
print("Precision:", test_metrics.box.mp)
print("Recall:", test_metrics.box.mr)

print("\n=== For comparison: same model's VALID metrics (from selection above) ===")
valid_row = comparison_df.iloc[0]
print("mAP50:", valid_row["mAP50"])
print("mAP50-95:", valid_row["mAP50-95"])
print("Precision:", valid_row["precision"])
print("Recall:", valid_row["recall"])

### Model Error Report
This section generates a comprehensive report of model errors observed during evaluation, including:
-   **Person Count Mismatches**: Images where the predicted number of 'person' instances differs from the ground truth.
-   **Overlapping Detections**: Instances where the model predicts both 'person' and 'people_wheelchair' with significant overlap (IoU > 0.5) on the same object.

This uses the same best-run weights (`BEST_WEIGHTS`) selected by the comparison cell earlier, so the CSV stays consistent with whichever model is ultimately exported. The report is provided in a downloadable CSV file for external analysis by another AI or tool.

In [ ]:
import pandas as pd
from ultralytics import YOLO
from pathlib import Path
import os
import yaml
from google.colab import files

# Helper function for IoU calculation (copied from earlier in the notebook)
def box_iou(box1, box2):
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

# Use the run selected as best in the comparison cell above (falls back to
# train-4 only if the comparison cell was not run, so this still works standalone)
best_weights_path = Path(BEST_WEIGHTS) if "BEST_WEIGHTS" in dir() else Path("runs/detect/train-4/weights/best.pt")

# Re-load dataset and data_cfg if they might not be in scope
# (Assuming they are available from prior cell executions, e.g., d4d10a20 and 4a633199)
# If running this cell independently, ensure 'dataset' and 'data_cfg' are defined.
# Example re-initialization (uncomment if needed):
# import roboflow
# ROBOFLOW_API_KEY = "wBuO0Xm5iCujzAp7ZaZj"
# ROBOFLOW_WORKSPACE = "first-group-project"
# ROBOFLOW_PROJECT = "wheelchair-9qvfx-bchvo"
# ROBOFLOW_MODEL_FORMAT = "yolov8"
# rf = roboflow.Roboflow(api_key=ROBOFLOW_API_KEY)
# project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
# latest_version_number = max([v.version for v in project.versions()])
# version = project.version(latest_version_number)
# dataset = version.download(ROBOFLOW_MODEL_FORMAT)
# with open(Path(dataset.location) / "data.yaml") as f:
#     data_cfg = yaml.safe_load(f)

if not best_weights_path.exists():
    print(f"Error: Best weights not found at {best_weights_path}. Cannot perform error analysis.")
else:
    model = YOLO(str(best_weights_path))
    valid_images_dir = Path(dataset.location) / "valid" / "images"
    valid_labels_dir = Path(dataset.location) / "valid" / "labels"
    names = data_cfg["names"]
    person_id = names.index("person")

    # --- 1. Gather Count Mismatches (Over/Under-predictions for 'person') ---
    print("Collecting person count mismatches...")
    results_predict = model.predict(
        source=str(valid_images_dir),
        conf=0.15,
        iou=0.6,
        save=False,
        verbose=False
    )

    all_errors_data = []

    for r in results_predict:
        img_path = Path(r.path)
        label_path = valid_labels_dir / (img_path.stem + ".txt")

        # Ground truth count for 'person'
        gt_person_count = 0
        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    if line.strip():
                        cls_id = int(line.split()[0])
                        if cls_id == person_id:
                            gt_person_count += 1

        # Predicted count for 'person'
        pred_person_count = sum(1 for c in r.boxes.cls if int(c) == person_id)

        diff = pred_person_count - gt_person_count
        if diff != 0: # Only record actual mismatches
            all_errors_data.append({
                "Error Type": "Person Count Mismatch",
                "Image Name": img_path.name,
                "GT Person Count": gt_person_count,
                "Predicted Person Count": pred_person_count,
                "Difference": diff,
                "Class 1": None, "Confidence 1": None,
                "Class 2": None, "Confidence 2": None,
                "IoU": None
            })

        # --- 2. Gather Overlapping Detections (person <-> people_wheelchair) ---
        boxes = r.boxes.xyxy.cpu().numpy()
        classes = r.boxes.cls.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()

        n = len(boxes)
        for i in range(n):
            for j in range(i + 1, n):
                c1_name, c2_name = names[classes[i]], names[classes[j]]
                # Filter for the specific confusion case
                if {c1_name, c2_name} == {"person", "people_wheelchair"}:
                    iou = box_iou(boxes[i], boxes[j])
                    if iou > 0.5: # Only record significant overlaps
                        all_errors_data.append({
                            "Error Type": "Overlapping Detection",
                            "Image Name": Path(r.path).name,
                            "GT Person Count": None, "Predicted Person Count": None, "Difference": None,
                            "Class 1": c1_name, "Confidence 1": float(confs[i]),
                            "Class 2": c2_name, "Confidence 2": float(confs[j]),
                            "IoU": float(iou)
                        })

    if all_errors_data:
        errors_df = pd.DataFrame(all_errors_data)
        output_filename = "model_error_report.csv"
        errors_df.to_csv(output_filename, index=False)

        print(f"\nError report saved to {output_filename}")
        print("First 5 rows of the error report:")
        display(errors_df.head())
        files.download(output_filename)
    else:
        print("\nNo significant errors found to report.")

---
# Project Summary & Conclusions

## Goal
Fine-tune a YOLOv8 object-detection model to detect three classes — `person`, `wheelchair`, and `people_wheelchair` — for an accessibility/safety use-case where **missing a real case is worse than a false alarm** (hence `conf=0.15`).

## Dataset for this run
**Roboflow** `first-group-project` / `wheelchair-9qvfx-bchvo` (see the "Download Dataset" cell for the exact version downloaded this run — YOLOv8 format), with built-in `train` / `valid` / `test` splits. All training and evaluation cells use `{dataset.location}/data.yaml`. Custom `test_clean` construction from older notebooks is **not** used. Every training run requests `epochs=30` with early stopping via `patience` (an earlier draft's markdown mentioned 150-epoch runs, but no code cell actually uses 150 — the code is authoritative).

**The `test` split is never loaded, evaluated, or predicted on during data prep, training, hyperparameter selection, or the run-comparison table.** Every one of those steps uses `valid` only, so the choice of "best model" is never informed by `test`. The single exception is the dedicated "Final Held-Out Check on the Test Split" cell placed right after model export, which evaluates the already-selected model on `test` exactly once, purely to report a genuine generalization estimate.

## What Was Done

**1. Data preparation & audit.** Download the dataset from Roboflow, verify `data.yaml` and split folders, check class distribution / empty labels / visual annotations on the native `valid` split, and visualize bounding-box outliers found by the IQR check.

**2. Baseline.** A 30-epoch `yolov8s` run validates the pipeline before long training.

**3. Systematic hyperparameter search (one parameter at a time)**, followed by additional architecture/optimizer experiments once the sweep plateaued:

| Run | Change | Role |
|---|---|---|
| Attempt 1 | `yolov8m`, `copy_paste=0.4` | Full model + rare-class augmentation |
| Attempt 2 | `copy_paste=0.5` | Augmentation sweep |
| Attempt 3 (train-4) | `cls=0.8` | Target class confusion |
| Attempt 4 | `cls=1.0` | Bracket the `cls` optimum |
| Freeze/Unfreeze | Two-phase transfer learning | Required technique check |
| SGD + `lr0=0.001` | Explicit optimizer & learning rate | LR actually takes effect (`optimizer=auto` silently ignores `lr0`) |
| SGD + `lr0=0.0005` + cosine LR | Lower LR, cosine schedule | Test whether an even lower, decaying LR helps further |
| SGD + extra augmentation | `mixup`, wider `scale`, stronger HSV jitter | Target remaining recall gaps with harder training examples |

**4. Model selection.** Rather than naming a winner here, a dedicated comparison cell evaluates **every** run above (`model.val(split="valid")`, identical `conf`/`iou` settings) and selects the one with the highest mAP50-95. See that cell's output (`comparison_df`) for the actual ranking on this run of the notebook — it can change between re-runs since training is stochastic, so no run name is hard-coded in this summary.

**5. Error analysis.** For the automatically-selected best run: inspect person over/under-predictions and person↔people_wheelchair overlapping detections on the `valid` images, showing every flagged case (not just a sample), to separate model error from annotation gaps.

## Next Steps
1. Review the full comparison table and the failure-case visualizations before trusting any single metric.
2. If person errors concentrate in dense crowds, audit train labels the same way (see the outlier/failure visualizations above for candidates).
3. Re-run the comparison cell whenever a new training experiment is added — it will automatically re-rank and can select a new best run without further edits.
4. ~~Only once a final model is genuinely settled on, evaluate it on the untouched `test` split exactly once, in a separate step outside this notebook's model-selection flow.~~ **Done** — see the "Final Held-Out Check on the Test Split" cell right after model export, which runs `BEST_WEIGHTS` on `test` exactly once.

In [ ]:
import pandas as pd
from ultralytics import YOLO
from pathlib import Path
import os
import yaml
import cv2
import matplotlib.pyplot as plt
import numpy as np

# --- Re-initialize necessary components for standalone execution ---
# The error was caused by a hardcoded path to /content/wheelchair-4.
# We update it to the actual location found in the notebook: /content/wheelchair-5
try:
    dataset_location = dataset.location
except NameError:
    dataset_location = '/content/wheelchair-5'

# Re-load data_cfg
data_yaml_path = Path(dataset_location) / "data.yaml"
if not data_yaml_path.exists():
    raise FileNotFoundError(f"Could not find data.yaml at {data_yaml_path}")

with open(data_yaml_path) as f:
    data_cfg = yaml.safe_load(f)

names = data_cfg["names"]
person_id = names.index("person")

best_weights_path = Path(BEST_WEIGHTS) if "BEST_WEIGHTS" in dir() else Path("runs/detect/train-4/weights/best.pt")
if not best_weights_path.exists():
    print(f"Error: Best weights not found at {best_weights_path}. Please ensure the training runs have completed.")
    model = None
else:
    model = YOLO(str(best_weights_path))

valid_images_dir = Path(dataset_location) / "valid" / "images"
valid_labels_dir = Path(dataset_location) / "valid" / "labels"

# Load the error report
output_filename = "model_error_report.csv"
if not Path(output_filename).exists():
    print(f"Error: Error report '{output_filename}' not found. Please run the 'Model Error Report' cell first.")
    errors_df = pd.DataFrame()
else:
    errors_df = pd.read_csv(output_filename)

# --- Helper functions ---
def box_iou(box1, box2):
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

def draw_text_with_bg(img, text, pt, color, thickness=2, font_scale=0.6, bg_color=(0, 0, 0)):
    font = cv2.FONT_HERSHEY_SIMPLEX
    (text_w, text_h), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    x, y = pt
    cv2.rectangle(img, (x, y - text_h - 4), (x + text_w + 4, y + baseline), bg_color, -1)
    cv2.putText(img, text, (x + 2, y - 2), font, font_scale, color, thickness)

# --- Visualizing Person Count Mismatches ---
print("\n--- Visualizing Person Count Mismatches ---")
person_mismatches = errors_df[errors_df["Error Type"].str.contains("Count Mismatch", na=False)].copy()
if not person_mismatches.empty:
    person_mismatches["Abs_Difference"] = person_mismatches["Difference"].abs()
    person_mismatches = person_mismatches.sort_values(by="Abs_Difference", ascending=False)
    sample_mismatches = person_mismatches.head(6)

    if model is not None:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        for plot_idx, (idx, row) in enumerate(sample_mismatches.iterrows()):
            img_name = row["Image Name"]
            img_path = valid_images_dir / img_name
            label_path = valid_labels_dir / (img_path.stem + ".txt")
            img = cv2.imread(str(img_path))
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            h, w = img.shape[:2]
            if label_path.exists():
                with open(label_path, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if parts and int(parts[0]) == person_id:
                            _, xc, yc, bw, bh = map(float, parts)
                            cv2.rectangle(img, (int((xc-bw/2)*w), int((yc-bh/2)*h)), (int((xc+bw/2)*w), int((yc+bh/2)*h)), (0, 255, 0), 2)
            results_img = model.predict(source=str(img_path), conf=0.15, iou=0.6, verbose=False)[0]
            for b in results_img.boxes:
                if int(b.cls) == person_id:
                    box = b.xyxy[0].cpu().numpy().astype(int)
                    cv2.rectangle(img, (box[0], box[1]), (box[2], box[3]), (255, 0, 0), 2)
            axes[plot_idx].imshow(img)
            axes[plot_idx].set_title(f"{img_name}\nGT: {row['GT Person Count']}, Pred: {row['Predicted Person Count']}")
            axes[plot_idx].axis("off")
        plt.tight_layout()
        plt.show()

# --- Visualizing Overlapping Detections ---
print("\n--- Visualizing Overlapping Detections ---")
overlap_detections = errors_df[errors_df["Error Type"].str.contains("Overlap", na=False)].copy()
if not overlap_detections.empty:
    overlap_detections = overlap_detections.sort_values(by="IoU", ascending=False)
    sample_overlaps = overlap_detections.head(6)
    if model is not None:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        for plot_idx, (idx, row) in enumerate(sample_overlaps.iterrows()):
            img_path = valid_images_dir / row["Image Name"]
            img = cv2.imread(str(img_path))
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[plot_idx].imshow(img)
            axes[plot_idx].set_title(f"{row['Image Name']}\nIoU={row['IoU']:.2f}")
            axes[plot_idx].axis("off")
        plt.tight_layout()
        plt.show()